# 3D NIfTI Reptile Few-Shot Pipeline

This notebook uses `config.py`, `resize.py`, `split.py`, `dataloader_ben2.py`, and `model.py`.

## Section 0.1: Install Dependencies


In [ ]:
import sys

INSTALL_DEPS = False

if INSTALL_DEPS:
    !{sys.executable} -m pip install nibabel scikit-image tqdm matplotlib numpy torch
else:
    print("Skipping dependency install. Set INSTALL_DEPS = True if imports fail.")


## Section 1: Read Config

In [ ]:
import config

print("Data directory:", config.DATA_DIR)
print("Resized data directory:", config.RESIZED_DATA_DIR)
print("Resize dimensions:", config.RESIZE_DIMS)
print("Batch size:", config.BATCH_SIZE)
print("UNet features:", config.UNET_FEATURES)
print("Reptile outer steps:", config.REPTILE_OUTER_STEPS)
print("Reptile inner steps:", config.REPTILE_INNER_STEPS)
print("Reptile inner LR:", config.REPTILE_INNER_LR)
print("Reptile outer LR:", config.REPTILE_OUTER_LR)

## Section 2: Initialise Functions

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import torch

from model import UNet3D, bce_dice_loss, dice_score, prepare_episode, validate, test, get_device
from dataloader_ben2 import build_3d_dataloader, build_episode_loader


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = get_device()
print("Python:", sys.executable)
print("Device:", device)

## Section 3: Resize Dataset And Split

In [ ]:
RUN_RESIZE = True

if RUN_RESIZE:
    !{sys.executable} resize.py --input-dir {config.DATA_DIR} --output-dir {config.RESIZED_DATA_DIR} --x {config.RESIZE_DIMS[0]} --y {config.RESIZE_DIMS[1]} --depth {config.RESIZE_DIMS[2]}
else:
    print("Skipping resize. Set RUN_RESIZE = True to regenerate data-resize/.")

In [ ]:
RUN_SPLIT = True

if RUN_SPLIT:
    !{sys.executable} split.py --data-dir {config.RESIZED_DATA_DIR} --output-dir . --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15 --seed 42
else:
    print("Skipping split generation.")

## Section 4: Reptile

In [ ]:
model = UNet3D().to(device)
print(model.__class__.__name__)

In [ ]:
SHOT_LIST = [1, 3, 5]
N_QUERY = 1
EVAL_EPISODES = 20
RUN_TRAINING = True

if RUN_TRAINING:
    for n_shot in SHOT_LIST:
        print(f"Training Reptile {n_shot}-shot")
        !{sys.executable} train_reptile.py --data-dir {config.RESIZED_DATA_DIR} --train-split train.txt --val-split val.txt --test-split test.txt --n-support {n_shot} --n-query {N_QUERY} --outer-steps {config.REPTILE_OUTER_STEPS} --inner-steps {config.REPTILE_INNER_STEPS} --inner-lr {config.REPTILE_INNER_LR} --outer-lr {config.REPTILE_OUTER_LR} --val-interval {config.REPTILE_VAL_INTERVAL} --batch-size {config.BATCH_SIZE} --eval-episodes {EVAL_EPISODES} --model-path reptile_{n_shot}shot.pth --history-path reptile_{n_shot}shot_history.json --metrics-path reptile_{n_shot}shot_metrics.json
else:
    print("Skipping training. Set RUN_TRAINING = True to train 1/3/5-shot Reptile models.")


In [ ]:
import json

SHOT_LIST = [1, 3, 5]
SELECTED_SHOT = 5

histories = {}
metrics_by_shot = {}
for n_shot in SHOT_LIST:
    history_path = Path(f"reptile_{n_shot}shot_history.json")
    metrics_path = Path(f"reptile_{n_shot}shot_metrics.json")

    histories[n_shot] = {"support_loss": [], "query_loss": [], "val_dice": []}
    if history_path.exists():
        histories[n_shot] = json.loads(history_path.read_text())

    metrics_by_shot[n_shot] = {
        "validation": {"dice": float("nan"), "loss": float("nan")},
        "test": {"dice": float("nan"), "loss": float("nan")},
    }
    if metrics_path.exists():
        metrics_by_shot[n_shot] = json.loads(metrics_path.read_text())

model_path = Path(f"reptile_{SELECTED_SHOT}shot.pth")
model_loaded = model_path.exists()
if model_loaded:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    print(f"Loaded {SELECTED_SHOT}-shot model:", model_path)
else:
    print(f"No saved {SELECTED_SHOT}-shot model found yet.")

history = histories.get(SELECTED_SHOT, {"support_loss": [], "query_loss": [], "val_dice": []})
val_metrics = metrics_by_shot.get(SELECTED_SHOT, {}).get("validation", {})
test_metrics = metrics_by_shot.get(SELECTED_SHOT, {}).get("test", {})

print("Selected validation:", val_metrics)
print("Selected test:", test_metrics)


## Section 5: Metrics

In [ ]:
import matplotlib.pyplot as plt

results = metrics_by_shot

print("=" * 52)
print(f"{'Shot':<10} {'Method':<14} {'Val Dice':>10} {'Test Dice':>10}")
print("=" * 52)
for n_shot in SHOT_LIST:
    val_dice = metrics_by_shot[n_shot]["validation"].get("dice", float("nan"))
    test_dice = metrics_by_shot[n_shot]["test"].get("dice", float("nan"))
    print(f"{str(n_shot) + '-shot':<10} {'Reptile':<14} {val_dice:>10.3f} {test_dice:>10.3f}")
print("=" * 52)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plotted = False
colours = {1: "steelblue", 3: "mediumseagreen", 5: "darkorange"}

for n_shot in SHOT_LIST:
    shot_history = histories.get(n_shot, {})
    values = np.array(shot_history.get("query_loss", []), dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        continue
    window = max(1, len(values) // 100)
    smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(values)), smoothed, label=f"{n_shot}-shot query loss", color=colours.get(n_shot))
    plotted = True

if plotted:
    ax.set_xlabel("Outer step")
    ax.set_ylabel("Loss")
    ax.set_title("Reptile Meta-Training Query Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training history yet.")


In [ ]:
labels = [f"{n}-shot" for n in SHOT_LIST]
val_scores = [metrics_by_shot[n]["validation"].get("dice", np.nan) for n in SHOT_LIST]
test_scores = [metrics_by_shot[n]["test"].get("dice", np.nan) for n in SHOT_LIST]

x = np.arange(len(SHOT_LIST))
width = 0.36
fig, ax = plt.subplots(figsize=(9, 5))
val_bars = ax.bar(x - width / 2, val_scores, width, label="Validation", color="steelblue", alpha=0.85)
test_bars = ax.bar(x + width / 2, test_scores, width, label="Test", color="darkorange", alpha=0.85)

for bars in [val_bars, test_bars]:
    for bar in bars:
        height = bar.get_height()
        if np.isfinite(height):
            ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.3f}", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Dice Score")
ax.set_ylim(0, 1.0)
ax.set_title("Reptile 1/3/5-Shot Validation And Test Dice")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("shot_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import copy
import torch.optim as optim

RUN_VISUALISE = True


def visualise_reptile_prediction(base_model, n_support=1, n_query=3, inner_steps=config.REPTILE_INNER_STEPS, inner_lr=config.REPTILE_INNER_LR, threshold=0.3):
    if not model_loaded:
        print("Warning: no saved model was loaded. This prediction is from an untrained model.")

    episode = next(build_episode_loader(
        data_dir=config.RESIZED_DATA_DIR,
        split_file="test.txt",
        n_support=n_support,
        n_query=n_query,
        episodes=1,
    ))

    support_ids = episode["support"]["case_id"]
    query_ids = episode["query"]["case_id"]
    print("Support case(s):", support_ids)
    print("Query case(s):", query_ids)

    support_images, support_masks, query_images, query_masks = prepare_episode(episode, device)

    adapted_model = copy.deepcopy(base_model).to(device)
    adapted_model.train()
    optimizer = optim.SGD(adapted_model.parameters(), lr=inner_lr)

    for _ in range(inner_steps):
        optimizer.zero_grad()
        loss = bce_dice_loss(adapted_model(support_images), support_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), 1.0)
        optimizer.step()

    adapted_model.eval()
    with torch.no_grad():
        logits = adapted_model(query_images)
        probabilities = torch.sigmoid(logits).cpu()
        predictions = (probabilities > threshold).float()

    query_images = query_images.cpu()
    query_masks = query_masks.cpu()
    n_rows = min(n_query, query_images.shape[0])

    fig, axes = plt.subplots(n_rows, 4, figsize=(14, 3.6 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row in range(n_rows):
        case_id = query_ids[row]
        gt_volume = query_masks[row, 0]
        pred_volume = predictions[row, 0]
        prob_volume = probabilities[row, 0]

        slice_scores = gt_volume.sum(dim=(1, 2))
        slice_idx = int(torch.argmax(slice_scores)) if torch.max(slice_scores).item() > 0 else gt_volume.shape[0] // 2

        intersection = (pred_volume * gt_volume).sum()
        union = pred_volume.sum() + gt_volume.sum()
        dice = ((2.0 * intersection + 1e-6) / (union + 1e-6)).item()
        pred_fraction = pred_volume.mean().item()
        prob_min = prob_volume.min().item()
        prob_mean = prob_volume.mean().item()
        prob_max = prob_volume.max().item()

        image_slice = np.rot90(query_images[row, 0, slice_idx].numpy())
        mask_slice = np.rot90(gt_volume[slice_idx].numpy())
        prob_slice = np.rot90(prob_volume[slice_idx].numpy())
        pred_slice = np.rot90(pred_volume[slice_idx].numpy())

        axes[row, 0].imshow(image_slice, cmap="gray")
        axes[row, 0].set_title(f"Input\n{case_id} | slice {slice_idx}", fontsize=9)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(image_slice, cmap="gray")
        axes[row, 1].imshow(mask_slice, cmap="Reds", alpha=0.45)
        axes[row, 1].set_title(f"Ground Truth Overlay\n{case_id}", fontsize=9)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(prob_slice, cmap="magma", vmin=0.0, vmax=1.0)
        axes[row, 2].set_title(f"Probability\nmin {prob_min:.2f} mean {prob_mean:.2f} max {prob_max:.2f}", fontsize=9)
        axes[row, 2].axis("off")

        axes[row, 3].imshow(image_slice, cmap="gray")
        axes[row, 3].imshow(pred_slice, cmap="Blues", alpha=0.45)
        axes[row, 3].set_title(f"Prediction Overlay > {threshold:.2f}\nDice {dice:.3f} | FG {pred_fraction:.1%}", fontsize=9)
        axes[row, 3].axis("off")

    fig.suptitle(f"Adapted using support case(s): {', '.join(support_ids)}", fontsize=11)
    plt.tight_layout()
    plt.savefig("qualitative_reptile_predictions.png", dpi=150, bbox_inches="tight")
    plt.show()


if RUN_VISUALISE:
    visualise_reptile_prediction(model, n_support=SELECTED_SHOT, n_query=3, threshold=0.3)
else:
    print("Skipping qualitative prediction visualisation. Set RUN_VISUALISE = True to display query case IDs.")


In [ ]:
for n_shot in SHOT_LIST:
    val_dice = metrics_by_shot[n_shot]["validation"].get("dice", float("nan"))
    test_dice = metrics_by_shot[n_shot]["test"].get("dice", float("nan"))
    print(f"{n_shot}-shot | validation Dice: {val_dice:.3f} | test Dice: {test_dice:.3f}")
